In [ ]:
# 0) Colab: mount Drive + clone code + copy data về disk local
# Data upload 1 lần từ máy local (build_train.py in danh sách) vào DRIVE/ranker_data/:
#   ranker_data/datasets/{train.parquet, train_users.parquet, build_meta.json}
#   ranker_data/pools/{eval_val.parquet, eval_val_users.parquet}
#   ranker_data/item_vectors.npy
# ⚠ DISCIPLINE: KHÔNG upload eval_test* — test chấm LOCAL sau khi chốt model (eval.py).
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)
DRIVE_DIR = '/content/drive/MyDrive/recommender_train_colab'
assert os.path.isdir(DRIVE_DIR), 'thiếu folder recommender_train_colab (xem retriever/train.ipynb cell 0)'

REPO = 'https://github.com/CrocodixD/anime-recommender.git'
CODE = '/content/recommender'
if os.path.exists(CODE):
    !cd {CODE} && git pull
else:
    !git clone {REPO} {CODE}

# copy data Drive -> disk local (đọc parquet trên Drive FUSE rất chậm)
DATA_DIR = '/content/ranker_data'
if not os.path.exists(DATA_DIR):
    !cp -r "{DRIVE_DIR}/ranker_data" {DATA_DIR}
!ls -lh {DATA_DIR}/datasets {DATA_DIR}/pools
%cd {CODE}

In [ ]:
# 1) Imports (torch) + path code ranker/src (import flat) — notebook = NN DIN cloud only
# (LightGBM đã chuyển train LOCAL: venv/bin/python ranker/src/train_lgbm.py)
import sys, json, time, shutil
from pathlib import Path
import numpy as np, pandas as pd, torch
sys.path.insert(0, str(Path.cwd() / 'ranker' / 'src'))

import train_nn
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

DATA = Path('/content/ranker_data')
print(json.loads((DATA / 'datasets' / 'build_meta.json').read_text()))

In [ ]:
# 2) Persistence trên Drive: runs_ranker/<VERSION>/<run_name>/{model.txt|model.pt, row.json}
# Leaderboard ranker_runs.csv rebuild từ row.json (bền với run ngắt — pattern retriever cell 5/9).
DRIVE = Path(DRIVE_DIR)
VERSION = 'r1'                       # r1 = artifacts v5_hist64_ep2 + pool-matched candidates
RUNS_DIR = DRIVE / 'runs_ranker' / VERSION
RESULTS_CSV = DRIVE / 'ranker_runs.csv'
LOCAL_RUNS = Path('/content/runs_ranker_local')
RUNS_DIR.mkdir(parents=True, exist_ok=True)

def sync_and_board(run_name):
    """Sync run local -> Drive 1 lần rồi rebuild leaderboard."""
    shutil.copytree(LOCAL_RUNS / run_name, RUNS_DIR / run_name, dirs_exist_ok=True)
    rows = [json.loads(p.read_text()) for p in sorted((DRIVE / 'runs_ranker').glob('*/*/row.json'))]
    for p, r in zip(sorted((DRIVE / 'runs_ranker').glob('*/*/row.json')), rows):
        r['version'] = p.parent.parent.name
    df = pd.DataFrame(rows).sort_values('val_ndcg@10', ascending=False)
    df.drop(columns=['importance_gain'], errors='ignore').to_csv(RESULTS_CSV, index=False)
    return df.drop(columns=['importance_gain'], errors='ignore')

print('runs ->', RUNS_DIR)

In [ ]:
# 5) Neural ranker (GPU) — so sánh với GBDT. Early stop trên two-stage val ndcg@10.
row = train_nn.train(DATA, LOCAL_RUNS, run_name='nn_din', epochs=2,
                     item_vectors=DATA / 'item_vectors.npy')
sync_and_board('nn_din')

# Knob: lr=3e-4 / batch_size=64 / hidden sâu hơn (sửa train_nn.NeuralRanker) — đổi run_name.

In [ ]:
# 6) Leaderboard (mọi run mọi version) — sort val_ndcg@10 @ best α.
# Chốt winner -> tải runs_ranker/<VERSION>/<run>/model.{txt,pt} về ranker/data/models/
# rồi LOCAL: eval.py (Pareto select + test report + val_cold) -> export.py.
rows = [json.loads(p.read_text()) | {'version': p.parent.parent.name}
        for p in sorted((DRIVE / 'runs_ranker').glob('*/*/row.json'))]
pd.DataFrame(rows).drop(columns=['importance_gain'], errors='ignore') \
  .sort_values('val_ndcg@10', ascending=False)